In [1]:
import gc, time, json, joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    average_precision_score, precision_recall_curve, roc_curve
)
import xgboost as xgb

In [2]:
t0 = time.time()
RANDOM_STATE = 42

In [3]:
dtypes = {
    "step": "int32", "type": "category", "amount": "float64",
    "nameOrig": "str", "oldbalanceOrg": "float64", "newbalanceOrig": "float64",
    "nameDest": "str", "oldbalanceDest": "float64", "newbalanceDest": "float64",
    "isFraud": "int8", "isFlaggedFraud": "int8",
}
df = pd.read_csv("fraud_det_dataset.csv", dtype=dtypes)
print(f"Loaded: {df.shape}  ({time.time()-t0:.1f}s)")

Loaded: (6362620, 11)  (105.4s)


In [4]:
df = df.drop_duplicates()
df = df.dropna()

In [5]:
# Balance-consistency errors: legitimate transactions satisfy
# oldbalance - amount = newbalance almost exactly; fraud often breaks this.
df["errorBalanceOrig"] = df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
df["errorBalanceDest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]

# Account-draining pattern: origin balance goes to exactly zero
df["origEmptiedOut"] = ((df["newbalanceOrig"] == 0) & (df["oldbalanceOrg"] > 0)).astype("int8")

# Destination was a dormant/zero account before receiving funds (classic mule pattern)
df["destWasZero"] = ((df["oldbalanceDest"] == 0) & (df["newbalanceDest"] == 0) & (df["amount"] > 0)).astype("int8")

# Merchant vs customer counterparty
df["isMerchantDest"] = df["nameDest"].str.startswith("M").astype("int8")

# Time features from `step` (1 step = 1 hour, simulation spans ~31 days)
df["hourOfDay"] = (df["step"] % 24).astype("int16")
df["dayOfSim"] = (df["step"] // 24).astype("int16")

# Ratio of transaction size to sender's balance (large fraction drained = suspicious)
df["amountToBalanceRatio"] = df["amount"] / (df["oldbalanceOrg"] + 1)

# One-hot encode transaction type
df = pd.get_dummies(df, columns=["type"], prefix="type", dtype="int8")

# Drop identifiers (high-cardinality, non-generalizable)
drop_cols = ["nameOrig", "nameDest"]
df = df.drop(columns=drop_cols)

feature_cols = [c for c in df.columns if c not in ("isFraud",)]
y_full = df["isFraud"].to_numpy(dtype="int8")
overall_fraud_rate = float(y_full.mean())
# Cast everything to float32 up front — halves memory vs float64, plenty of precision here
X_full = df[feature_cols].astype("float32").to_numpy()
del df
gc.collect()
print(f"Feature matrix: {X_full.shape}, fraud rate: {y_full.mean()*100:.4f}%  ({time.time()-t0:.1f}s)")


Feature matrix: (6362620, 20), fraud rate: 0.1291%  (236.1s)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=RANDOM_STATE, stratify=y_full
)
del X_full, y_full
gc.collect()
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train fraud count: {y_train.sum()}, Test fraud count: {y_test.sum()}")

Train: (5090096, 20), Test: (1272524, 20)
Train fraud count: 6570, Test fraud count: 1643


In [7]:
# logistic regression:

In [8]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype("float32")

t1 = time.time()
logreg = LogisticRegression(max_iter=200, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
logreg.fit(X_train_scaled, y_train)
print(f"LogReg trained in {time.time()-t1:.1f}s")
del X_train_scaled
gc.collect()

X_test_scaled = scaler.transform(X_test).astype("float32")
y_pred_lr = logreg.predict(X_test_scaled)
y_proba_lr = logreg.predict_proba(X_test_scaled)[:, 1]
del X_test_scaled
gc.collect()


C:\Users\Sejal Varma\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


LogReg trained in 79.1s


0

In [9]:
# xgboost

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
t2 = time.time()
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    eval_metric="aucpr",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train)
print(f"XGBoost trained in {time.time()-t2:.1f}s")
del X_train
gc.collect()

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]


XGBoost trained in 366.1s


In [10]:
# evaluating the models:

def evaluate(name, y_true, y_pred, y_proba):
    report = classification_report(y_true, y_pred, target_names=["Legit", "Fraud"], output_dict=True)
    cm = confusion_matrix(y_true, y_pred).tolist()
    roc_auc = roc_auc_score(y_true, y_proba)
    pr_auc = average_precision_score(y_true, y_proba)
    print(f"\n=== {name} ===")
    print(classification_report(y_true, y_pred, target_names=["Legit", "Fraud"]))
    print("Confusion matrix [[TN FP][FN TP]]:", cm)
    print(f"ROC-AUC: {roc_auc:.4f}   PR-AUC: {pr_auc:.4f}")
    return {"report": report, "confusion_matrix": cm, "roc_auc": roc_auc, "pr_auc": pr_auc}

results = {}
results["logreg"] = evaluate("Logistic Regression", y_test, y_pred_lr, y_proba_lr)
results["xgboost"] = evaluate("XGBoost", y_test, y_pred_xgb, y_proba_xgb)

# Feature importance (XGBoost)
importances = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nTop 10 feature importances (XGBoost):")
print(importances.head(10))


=== Logistic Regression ===
              precision    recall  f1-score   support

       Legit       1.00      0.99      1.00   1270881
       Fraud       0.12      1.00      0.21      1643

    accuracy                           0.99   1272524
   macro avg       0.56      0.99      0.60   1272524
weighted avg       1.00      0.99      0.99   1272524

Confusion matrix [[TN FP][FN TP]]: [[1258845, 12036], [7, 1636]]
ROC-AUC: 0.9993   PR-AUC: 0.9438

=== XGBoost ===
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00   1270881
       Fraud       1.00      1.00      1.00      1643

    accuracy                           1.00   1272524
   macro avg       1.00      1.00      1.00   1272524
weighted avg       1.00      1.00      1.00   1272524

Confusion matrix [[TN FP][FN TP]]: [[1270877, 4], [4, 1639]]
ROC-AUC: 0.9993   PR-AUC: 0.9984

Top 10 feature importances (XGBoost):
newbalanceOrig          0.340682
origEmptiedOut          0.300886
err

In [11]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba_xgb)
# find threshold achieving recall >= 0.90 with best precision at that recall
target_recall = 0.90
idxs = np.where(recalls[:-1] >= target_recall)[0]
if len(idxs) > 0:
    best_idx = idxs[np.argmax(precisions[:-1][idxs])]
    chosen_threshold = thresholds[best_idx]
    print(f"\nThreshold for recall>=90%: {chosen_threshold:.4f} "
          f"(precision={precisions[best_idx]:.4f}, recall={recalls[best_idx]:.4f})")
else:
    chosen_threshold = 0.5


Threshold for recall>=90%: 0.9999 (precision=1.0000, recall=0.9976)


In [14]:
# Save ROC / PR curve data + importances + results for reporting

importances.to_csv("feature_importances.csv")

with open("results.json", "w") as f:
    json.dump({
        "logreg": {"roc_auc": results["logreg"]["roc_auc"], "pr_auc": results["logreg"]["pr_auc"],
                   "report": results["logreg"]["report"], "confusion_matrix": results["logreg"]["confusion_matrix"]},
        "xgboost": {"roc_auc": results["xgboost"]["roc_auc"], "pr_auc": results["xgboost"]["pr_auc"],
                    "report": results["xgboost"]["report"], "confusion_matrix": results["xgboost"]["confusion_matrix"]},
        "chosen_threshold_90recall": float(chosen_threshold),
        "n_train": int(len(y_train)), "n_test": int(len(y_test)),
        "fraud_rate": overall_fraud_rate,
    }, f, indent=2)
